<!-- NB_DOC_INTRO_V1 -->
# Data Engineering — PySpark

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Apache Spark** = moteur distribue pour traiter des donnees qui ne tiennent pas en memoire d'une machine. **PySpark** = binding Python.

Ce notebook couvre l'API en profondeur avec **code execute en local single-node** (Spark sait tourner sur 1 machine, parfait pour developper). Pour multi-node (Databricks, EMR, K8s), meme code, juste config differente.

⚠️ **Pre-requis** : Java 11+ installe sur le systeme (Spark = JVM). Sur Mac : `brew install openjdk@17`. Sur Windows : verifier `JAVA_HOME`.

**Quand utiliser Spark ?** Honnetement, en 2025 : **pas pour < 1 TB**. Polars + DuckDB single-machine sont souvent suffisants et plus rapides. Spark s'impose pour vrai multi-node ou pipelines Databricks existants.

Versions : `pyspark >= 3.5`, Java >= 11.

## Plan

1. Quand utiliser Spark vs Polars vs DuckDB
2. Concepts cles (RDD, DataFrame, transformation/action, partition, shuffle, Catalyst)
3. **Demo executable** : SparkSession + DataFrame operations
4. Spark SQL
5. Joins et shuffles (broadcast, salting)
6. Pandas UDF (vectorise via Arrow)
7. MLlib (ML distribue)
8. Bonnes pratiques performance
9. Comparatif PySpark vs Polars vs DuckDB
10. Pieges et anti-patterns
11. References


## 1. Quand utiliser Spark ?

| Volume | Outil recommande 2025 |
|---|---|
| < 100k lignes | pandas |
| 100k - 10M lignes | **pandas** ou **polars** (single machine) |
| 10M - 1B lignes | **polars** (RAM ok) ou **DuckDB** ou **Spark single-node** |
| 1B+ lignes, multi-node | **Spark** (Databricks, EMR, GKE, DataProc) |
| TB+ continuous streaming | **Spark Structured Streaming** ou **Flink** |

> 💡 **Avant Spark** : essaye polars / DuckDB. Souvent 95% des cas "big data" tiennent dans un seul gros server moderne (64-512 GB RAM).

## 2. Concepts cles

| Notion | Sens |
|---|---|
| **RDD** (Resilient Distributed Dataset) | API bas niveau, immuable, distribue |
| **DataFrame** | API haut niveau (= pandas DataFrame distribue) |
| **Dataset** | DataFrame type (Scala/Java surtout) |
| **Transformation** | Operation **lazy** (`select`, `filter`, `groupBy`) |
| **Action** | Trigger l'execution (`show`, `collect`, `count`, `write`) |
| **Partition** | Sous-ensemble de la data sur 1 worker |
| **Shuffle** | Redistribution entre workers (couteux !) — a EVITER |
| **Catalyst** | Optimiseur de requetes (similaire a DuckDB / Polars optimiseur) |
| **Tungsten** | Moteur d'execution memoire-optimise |
| **AQE** (Adaptive Query Execution) | Re-optim a la volee |


## 3. Demo executable — SparkSession + DataFrame


In [ ]:
# pip install -q pyspark
import os
os.environ["PYSPARK_PYTHON"] = "python"   # eviter conflits Python sur Windows

try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

    spark = (
        SparkSession.builder
        .appName("DemoNotebook")
        .config("spark.sql.shuffle.partitions", "4")        # default 200, on baisse pour single-node
        .config("spark.sql.adaptive.enabled", "true")       # AQE
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print(f"Spark version : {spark.version}")
    print(f"Cores         : {spark.sparkContext.defaultParallelism}")
    SPARK_OK = True
except Exception as e:
    print(f"PySpark indisponible : {type(e).__name__}")
    print(f"Cause probable : Java pas installe (Spark requiert JVM)")
    print(f"Sur Mac : brew install openjdk@17")
    print(f"Le notebook continue en pseudo-code uniquement.")
    SPARK_OK = False

In [ ]:
# === Creer un DataFrame from scratch ===
if SPARK_OK:
    import numpy as np
    np.random.seed(42)
    n = 10000

    data = [
        (i,
         np.random.choice(["A", "B", "C", "D"]),
         float(np.random.exponential(100)),
         int(np.random.poisson(5)))
        for i in range(n)
    ]
    schema = StructType([
        StructField("id", IntegerType()),
        StructField("category", StringType()),
        StructField("amount", DoubleType()),
        StructField("qty", IntegerType()),
    ])
    df = spark.createDataFrame(data, schema)
    df.printSchema()
    df.show(5)
    print(f"Count : {df.count()}")
else:
    print("Demo skip — Spark non disponible.")

In [ ]:
# === DataFrame API : select, filter, agg ===
if SPARK_OK:
    # Select + expression
    df.select(
        F.col("category"),
        (F.col("amount") * 1.2).alias("amount_ttc"),
    ).show(5)

    # Filter
    print("Lignes avec amount > 200 et category A/B :")
    (df.filter((F.col("amount") > 200) & F.col("category").isin(["A", "B"]))
       .count())

    # GroupBy + agg
    (df
     .groupBy("category")
     .agg(
         F.sum("amount").alias("total"),
         F.avg("amount").alias("mean"),
         F.count("*").alias("n"),
         F.countDistinct("qty").alias("qty_unique"),
     )
     .orderBy(F.desc("total"))
     .show())
else:
    print("Skip.")

## 4. Spark SQL — souvent plus lisible


In [ ]:
if SPARK_OK:
    # Enregistrer la DF comme vue temporaire
    df.createOrReplaceTempView("transactions")

    result = spark.sql('''
        SELECT category,
               COUNT(*) AS n,
               SUM(amount) AS total,
               AVG(amount) AS mean,
               PERCENTILE_APPROX(amount, 0.95) AS p95
        FROM transactions
        WHERE amount > 50
        GROUP BY category
        ORDER BY total DESC
    ''')
    result.show()
else:
    print("Skip.")

## 5. Joins et shuffles

### Types de joins
```python
df1.join(df2, on="id", how="inner")     # inner, left, right, outer, semi, anti, cross
df1.join(df2, df1.id == df2.user_id, "left")
```

### Broadcast join (anti-shuffle)
Si une table est petite (< 10 MB), la **broadcaster** sur tous les workers → evite le shuffle de la grande table.
```python
from pyspark.sql.functions import broadcast
df_big.join(broadcast(df_small), "id")
# → evite le shuffle de df_big (qui coute cher)
```

### Skew (cle desequilibree)
Si une cle represente 90% des donnees, le worker qui la traite est noye.
Solutions :
- `spark.sql.adaptive.skewJoin.enabled = true` (AQE auto)
- **Salting** : ajouter du bruit a la cle
```python
from pyspark.sql.functions import concat, lit, rand
df.withColumn("key_salted",
    F.concat(F.col("key"), F.lit("_"), (F.rand() * 10).cast("integer")))
```


## 6. Pandas UDF (vectorise via Arrow)

### UDF Python classique (LENT — eviter)
```python
@F.udf(returnType=DoubleType())
def my_func(x):
    return x * 2
df.withColumn("y", my_func("x"))
# → serialisation Python pour CHAQUE row, lent
```

### Pandas UDF (RECOMMANDE)
Vectorise via Arrow, ~ 100× plus rapide.
```python
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def normalize(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

df.withColumn("amount_norm", normalize("amount")).show()
```

### GroupBy + Pandas UDF (applyInPandas)
```python
@pandas_udf("user_id long, mean_amount double")
def per_user_stats(pdf: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "user_id":     [pdf["user_id"].iloc[0]],
        "mean_amount": [pdf["amount"].mean()],
    })

df.groupBy("user_id").applyInPandas(per_user_stats, schema="...").show()
```


## 7. MLlib — ML distribue dans Spark

```python
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=["x1", "x2", "x3"], outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features")
clf = RandomForestClassifier(labelCol="y", featuresCol="features", numTrees=100)

pipeline = Pipeline(stages=[assembler, scaler, clf])
model = pipeline.fit(train_df)
preds = model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol="y").evaluate(preds)
print(f"AUC : {auc:.4f}")
```

> Note : **MLlib < sklearn** en richesse. Pour ML serieux on entraine souvent sur sample en pandas/sklearn, et on utilise Spark juste pour le scoring batch.


## 8. Bonnes pratiques performance

| Astuce | Effet |
|---|---|
| `spark.read.parquet` au lieu de CSV | × 5-10 lecture |
| Filtres AVANT joins | Reduit le shuffle |
| `cache()` ou `persist()` sur DF reutilise | Evite recomputes |
| `broadcast(small_df)` | Evite shuffle |
| `coalesce(n)` avant write | Evite mille petits fichiers output |
| `repartition(n)` avant gros traitement | Parallelisme controle |
| Eviter `collect()` sur gros DF | OOM driver |
| Eviter UDF Python — utiliser Pandas UDF | ~ 100× speedup |
| Activer AQE | Auto-tuning shuffle / skew |
| Partition output par date | Predicate pushdown |
| `spark.sql.adaptive.coalescePartitions.enabled` | Reduit nb fichiers small |


## 9. Comparatif PySpark vs Polars vs DuckDB

| Critere | PySpark | Polars | DuckDB |
|---|---|---|---|
| Setup | JVM lourd | Trivial | Trivial |
| Perf single-machine | Mediocre vs polars | Excellent | Excellent |
| Multi-machine | ✅ | ❌ | ❌ (en cours) |
| SQL | ✅ | Partiel | ✅✅✅ |
| API DataFrame | OK | Tres bonne | N/A |
| Memoire requise | Lourd (driver + executors) | Leger | Tres leger |
| Lazy / Optimiseur | Catalyst | Polars optimiser | DuckDB optimiser |
| Streaming | ✅ Structured Streaming | Limite | Limite |
| Ecosysteme ML | MLlib (limite) | Via interop | Via interop |
| Cloud-native | Databricks, EMR, GKE, DataProc | Local only | DuckDB Cloud (jeune) |

> 🎯 **2025 recommendation** :
> - **< 1 TB sur 1 machine** → DuckDB ou Polars
> - **Pipelines Databricks existants** → Spark (continue)
> - **Vraie distribution multi-node** (PB+) → Spark
> - **Streaming temps reel** → Spark Structured Streaming ou Flink


## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| `df.toPandas()` sur 100M rows | OOM driver — utiliser pour exploration legere uniquement |
| Boucle `for` sur les rows | C'est pour ca que Spark existe... |
| Trop de shuffles (join multi-cle sans broadcast) | Broadcast les petites tables, AQE |
| Petits fichiers en sortie (`coalesce(1)` pour 1 fichier final) | `repartition` avant write, sauf si exigence "1 fichier" |
| UDF Python partout | Pandas UDF ou expressions natives |
| `spark.sql.shuffle.partitions=200` par defaut sur petit cluster | Ajuster a `core × 2-4` |
| Pas de schema explicite a la lecture | Inference lente, errors-prone |
| `cache()` sans `unpersist()` | Memoire qui s'accumule |
| Trop de stages | Repenser le pipeline |
| Recreer SparkSession par notebook | Garder une seule session globale |

## 11. References

### Documentation officielle
- **PySpark** : https://spark.apache.org/docs/latest/api/python/
- **Spark SQL** : https://spark.apache.org/docs/latest/sql-programming-guide.html
- **MLlib** : https://spark.apache.org/docs/latest/ml-guide.html
- **Structured Streaming** : https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html

### Tutoriels et benchmarks
- *Databricks Academy* : https://www.databricks.com/learn/training
- *Spark The Definitive Guide* (Chambers & Zaharia)
- H2O / TPC-H benchmarks (compare PySpark / Polars / DuckDB)

Voir aussi :
- [BDD_DuckDB.ipynb](./BDD_DuckDB.ipynb) — alternative single-node
- [Structures_Polars.ipynb](./Structures_Polars.ipynb) — alternative single-node
- [MLOps_Pipelines_Airflow_Prefect.ipynb](./MLOps_Pipelines_Airflow_Prefect.ipynb) — orchestrer Spark


In [ ]:
# === Cleanup ===
if SPARK_OK:
    spark.stop()
    print("Spark stopped.")
else:
    print("Rien a cleanup.")